In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import os.path as osp

from pathlib import Path
from pprint import pprint

import sys

root = Path.cwd().parent 
if str(root) not in sys.path:    
    sys.path.append(str(root))

In [3]:
from src.config import CFG

import warnings
warnings.filterwarnings(action="ignore", category=Warning)

In [4]:
from langchain_openai import ChatOpenAI

In [5]:
llm = ChatOpenAI(
    base_url=f"http://localhost:{CFG.MODEL_PORT}/v1",
    api_key="not-needed",
    model="local-model"
)


In [6]:
Q = "Explain Docker in one sentence."

In [7]:
%%time
response = llm.invoke(input=Q)

CPU times: user 48.9 ms, sys: 5.59 ms, total: 54.5 ms
Wall time: 28.1 s


In [8]:
response.content

'Docker is a platform that packages applications and all their dependencies into portable, standardized units called containers, ensuring they run reliably and consistently across any computing environment.'

In [9]:
llm = ChatOpenAI(
    base_url=f"http://localhost:{CFG.MODEL_PORT}/v1",
    api_key="not-needed",
    model="local-model",
    streaming=CFG.STREAMING,
    extra_body={
        "reasoning_effort": "low",
        "chat_template_kwargs": {"enable_thinking": True},
    }
)


In [10]:
%%time

response = ""
for chunk in llm.stream(input=Q):
    if chunk.content:
        print(chunk.content, end="", flush=True)
        response+=chunk.content

Docker is a platform that allows developers to package an application and all of its dependencies into standardized, portable units called containers, ensuring the software runs reliably the same way regardless of the environment.CPU times: user 300 ms, sys: 36.6 ms, total: 337 ms
Wall time: 31.6 s


## Testing African languages

In [11]:
%%time

kinyarwanda_Q = "Witwa nde?"

messages = [
    {
        "role": "system", 
        "content": """You are a multilingual assistant that can 
                respond in various languages inluding African ones.
                First, precise the language you are being prompted in 
                and answer in the same"""
    },
    {
        "role": "user",
        "content": kinyarwanda_Q
    }
]


response = ""
for chunk in llm.stream(input=messages):
    print(chunk.content, end="", flush=True)
    response+=chunk.content

print()

**Lingala.**

Nazali modélò ya luambi (AI) oyo ezali kosala makambo ya lingála. Nzali makasi kosalaka na yo.
CPU times: user 835 ms, sys: 159 ms, total: 994 ms
Wall time: 1min


In [12]:
messages.append({
    "role": "assistant",
    "content": response
})

In [13]:

kinyarwanda_Q = "mubyukuri, iyi yari kinyarwanda ntabwo yari kiswahili. Hari?"
messages.append(
    {
        "role": "user",
        "content": kinyarwanda_Q
    }
)

In [14]:
%%time

response = ""
for chunk in llm.stream(input=messages):
    print(chunk.content, end="", flush=True)
    response+=chunk.content

print()

**Kinyarwanda.**

Ndabwo uri binyuze, kandi ndakunda gusaba imbabazi ku bw'akoze igisubizo gashobora kuba biba byiza. Urakoze cyane ku kumenyesha!

Ni ngombwa yanjye kubaka ubuhanga bwanjye. Ubu, nshobora kugufasha gute mu Kinyarwanda cyangwa mu rurimi rugomba?
CPU times: user 836 ms, sys: 157 ms, total: 993 ms
Wall time: 53.4 s


In [15]:
messages.append({
    "role": "assistant",
    "content": response
})

In [16]:

kinyarwanda_Q = "how do we say sand in kinyarwanda?"
messages.append(
    {
        "role": "user",
        "content": kinyarwanda_Q
    }
)

In [17]:
%%time

response = ""
for chunk in llm.stream(input=messages):
    print(chunk.content, end="", flush=True)
    response+=chunk.content

print()

**English.**

The word for "sand" in Kinyarwanda is **umuciriro**.

If you are referring to very fine, small grains, you might also hear variations, but **umuciriro** is the most common and correct term.

*Example: *Umuciriro wa karere (Sand of the area)*
CPU times: user 494 ms, sys: 94.3 ms, total: 588 ms
Wall time: 32.6 s
